# Model V4 — 한우 도체 최종등급(LAST_GRADE) 예측 (통합·정리판)

이 노트북은 `축산 데이터 코드` / `hanwoo_eda` / `gender_split_model` / `model_v1_1` / `model_v1_1_model_compare` / `model_v3` 를
하나로 정리하고 발견된 문제를 모두 고친 **최종본**이다.

## 이전 버전 대비 개선점
1. **코드 중복 제거** — 흩어져 있던 전처리/모델 빌드 로직을 `hanwoo_pipeline.py` 모듈 하나로 통합. (버그 수정이 한 곳에서 끝남)
2. **경로 자동 탐지** — `BASE_DIR` 하드코딩(`C:\Users\010hy...`, `C:\Users\Administrator...`) 제거. 어느 PC에서든 동작.
3. **★ 타깃 누수 차단** — `WGRADE`(육량등급)는 `LAST_GRADE` 뒷글자 A/B/C 와 **96% 동일**하다.
   이전 모델(0.94~0.95)은 사실상 정답의 절반을 feature 로 넣은 셈. V4 는 기본적으로 `WGRADE` 를 제외하고,
   그 영향을 ablation 으로 정량화한다.
4. **기상 누수 차단** — rolling feature 는 `shift(1)` 후 계산(도축일 당일/미래 미포함). 관측소 좌표가 없으면 공간보간 자동 skip.
5. **train 기준 보간** — 모든 group 중앙값/최빈값은 train split 에서만 fit → valid/test 누수 차단.
6. **'수'(수소) 처리** — 표본 0.6% 로 너무 적어 분리 모델에서 제외, 통합 모델 예측으로 fallback.
7. **모델 확장** — RandomForest / ExtraTrees / HistGB / XGBoost / LightGBM (미설치 모델은 자동 skip).


## 0. 임포트 & 설정

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import hanwoo_pipeline as hp   # ← 공통 모듈 (같은 폴더)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 120)

# ───── 설정 ─────
# 빠른 확인은 USE_SAMPLE=True. 전체(약 240만 행) 재현은 USE_SAMPLE=False (수십 분 소요).
cfg = hp.Config(
    use_sample=True,
    sample_n=200_000,
    use_wgrade_feature=False,   # ★ 누수 차단. True 로 바꾸면 이전 버전과 동일(과대평가) 구성.
)
print("available models:", hp.available_models(cfg))
cfg

## 1. 데이터 로드
원본 5종(train/weather/area/death/lineage)을 자동 탐지 경로로 읽는다.

In [ ]:
t0 = time.time()
data = hp.load_raw(cfg)
print("base_dir:", data["base_dir"])
for k in ["train", "weather", "area", "death", "lineage"]:
    v = data.get(k)
    print(f"  {k:8s}: {None if v is None else v.shape}")
print(f"load: {time.time()-t0:.0f}s")

## 2. 특성 생성 + split

`build_feature_frame` 한 번에 다음이 모두 처리된다.
- 70/15/15 stratified split (`LAST_GRADE` 기준)
- 농장 면적/사육두수(`cow_year`, `log_area`), 폐사 수(`farm_death_count`), 혈통 KPN 빈도(`kpn_freq`) 병합
- 기상 30/90/180일 rolling 평균·합계(THI 포함, `shift(1)` 적용)
- **train split 기준** group 중앙값/최빈값 결측 보간 (WGRADE·도체측정·COST_AMT·면적 등)


In [ ]:
t0 = time.time()
parts, fitted = hp.build_feature_frame(cfg, data)
print("split shapes:", {k: v.shape for k, v in parts.items()})
print(f"feature build: {time.time()-t0:.0f}s")

# 등급 분포(train)
parts["train"][cfg.target].value_counts(normalize=True).sort_index().round(4)

## 3. 특성 목록 & 정책

`use_wgrade_feature=False` 이면 `WGRADE` 는 categorical 에서 빠진다.
도체 측정값(BACKFAT/REA/INSFAT/...)은 과제 정의상 사용하지만, 이들 역시 등급을 거의 결정하는
값이라는 점(아래 결론 참조)을 기억할 것.

In [ ]:
numf, catf = hp.get_feature_columns(cfg, parts["train"])
print(f"numeric ({len(numf)}):")
print(" ", numf)
print(f"\ncategorical ({len(catf)}): {catf}")
print(f"\nWGRADE 포함 여부: {cfg.use_wgrade_feature}  (False=누수 차단)")

## 4. 모델 비교 — 정직한 구성 (WGRADE 제외)

각 모델을 **unified**(전체 1모델) 와 **gender_split**(암/거세 분리, 수는 통합 fallback) 두 전략으로 학습·평가.
지표는 대회 기준 **Macro-F1** (16개 등급 평균).

In [ ]:
model_list = [m for m in ["random_forest", "lightgbm", "xgboost"] if m in hp.available_models(cfg)]
comp, preds, trained, (numf, catf) = hp.run_experiment(
    cfg, parts, model_names=model_list, strategies=("unified", "gender_split"))
comp.round(4)

In [ ]:
# 비교 막대그래프
fig, ax = plt.subplots(figsize=(10, 5))
piv = comp.pivot_table(index="model", columns="strategy", values="test_macro_f1")
piv.plot(kind="bar", ax=ax)
ax.set_ylabel("Test Macro-F1"); ax.set_title("모델 × 전략 Test Macro-F1 (WGRADE 제외)")
ax.set_ylim(max(0.5, piv.min().min() - 0.05), 1.0)
ax.legend(title="strategy"); ax.tick_params(axis="x", rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

## 5. ★ WGRADE 누수 ablation

같은 데이터·같은 LightGBM 으로 `WGRADE` 만 넣었다 뺐다 한 차이.
`WGRADE` 는 `LAST_GRADE` 의 육량등급(A/B/C)과 96% 동일하므로 **원칙적으로 제외하는 것이 맞다**(타깃 누출).

다만 실측 결과 그 *한계 효과는 작다*(아래 표에서 약 +0.005). 이유는 `WINDEX`(육량지수)가
이미 육량등급을 공식으로 결정하기 때문 — `WGRADE` 는 `WINDEX` 가 있으면 거의 중복이다.
즉 점수를 떠받치는 진짜 원인은 `WGRADE` 단독이 아니라 **도체 측정값 전체**(아래 결론 참조).

In [ ]:
cfg_wg = hp.Config(use_sample=cfg.use_sample, sample_n=cfg.sample_n, use_wgrade_feature=True)
comp_wg, preds_wg, _, _ = hp.run_experiment(
    cfg_wg, parts, model_names=["lightgbm"], strategies=("unified",))

ablation = pd.concat([
    comp[(comp.model == "lightgbm") & (comp.strategy == "unified")].assign(config="WGRADE 제외(정직)"),
    comp_wg.assign(config="WGRADE 포함(누수)"),
], ignore_index=True)[["config", "valid_macro_f1", "test_macro_f1"]]
ablation.round(4)

## 6. 최고 모델 상세 — 등급별 성능
특히 희소 등급(`등외`, `3C`, `3A`)의 F1 을 확인. Macro-F1 은 이들에 민감하다.

In [ ]:
best = comp.iloc[0]
best_key = f"{best['model']}_{best['strategy']}"
y_test = parts["test"][cfg.target].to_numpy()
y_pred = preds["test"][best_key]
print(f"BEST (WGRADE 제외): {best_key}  test Macro-F1 = {best['test_macro_f1']:.4f}")

rep = hp.per_class_report(y_test, y_pred)
rep[["precision", "recall", "f1-score", "support"]].round(3)

In [ ]:
# 등급별 F1 막대그래프
grade_rows = [i for i in rep.index if i not in ("accuracy", "macro avg", "weighted avg")]
f1s = rep.loc[grade_rows, "f1-score"].sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(f1s.index, f1s.values, color="steelblue")
ax.set_xlabel("F1-score"); ax.set_title(f"등급별 F1 — {best_key}")
ax.axvline(best["test_macro_f1"], color="red", ls="--", lw=1, label=f"Macro-F1={best['test_macro_f1']:.3f}")
ax.legend(); plt.tight_layout(); plt.show()

## 7. 성별별 Macro-F1
분리 학습이 실제로 도움이 되는지 성별 단위로 확인.

In [ ]:
hp.per_sex_macro_f1(y_test, y_pred, parts["test"]["JUDGE_SEX"]).round(4)

## 8. 혼동행렬
인접 등급(예: 1++B↔1+B)에서 혼동이 집중되는지 확인.

In [ ]:
labels = sorted(pd.unique(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("예측"); ax.set_ylabel("실제"); ax.set_title(f"혼동행렬 — {best_key}")
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 9. 특성 중요도 (tree 계열)

In [ ]:
best_model = trained.get(best["model"])
imp = None
try:
    mdl = best_model.named_steps["model"]
    pre = best_model.named_steps["preprocess"]
    names = list(numf)
    if "cat" in pre.named_transformers_:
        cat_t = pre.named_transformers_["cat"]
        if "onehot" in cat_t.named_steps:
            names += cat_t.named_steps["onehot"].get_feature_names_out(catf).tolist()
        else:
            names += catf
    fi = getattr(mdl, "feature_importances_", None)
    if fi is not None:
        n = min(len(names), len(fi))
        imp = pd.DataFrame({"feature": names[:n], "importance": fi[:n]}).sort_values("importance", ascending=False).head(20)
except Exception as e:
    print("importance 추출 실패:", e)

if imp is not None:
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(imp["feature"][::-1], imp["importance"][::-1], color="darkorange")
    ax.set_title(f"상위 20개 특성 중요도 — {best['model']}")
    plt.tight_layout(); plt.show()
    display(imp.reset_index(drop=True))

## 10. 결론 (200k 표본 실측 기준)

1. **`WGRADE` 는 타깃 누출이지만 한계 효과는 작다.** `LAST_GRADE` 육량등급과 96% 동일하므로 제외가 원칙.
   다만 빼도 LightGBM unified 가 0.979 → 포함 시 0.984 로 **약 +0.005** 차이뿐. `WINDEX` 가 이미
   육량등급을 결정해 `WGRADE` 가 중복이기 때문. (이전 노트북이 넣었던 건 잘못이지만 점수 거품의 주범은 아님)
2. **진짜 원인은 도체 측정값 전체.** 근내지방도 `INSFAT`(→육질) + `WINDEX`/`BACKFAT`/`REA`(→육량)가
   등급을 거의 결정한다. 즉 이 과제는 *"도체 측정값으로 등급 규칙을 재현"* 하는 문제에 가깝고,
   기상/농장/혈통의 기여는 작다. **만약 대회 의도가 도축 전(농장/기상/혈통만으로) 예측이라면 도체 측정값 전체가
   누출이며, 그때 난이도는 훨씬 높아진다** — 어떤 과제인지 먼저 확정해야 한다.
3. **부스팅 > 배깅.** 200k 에서 LightGBM 0.982 / XGBoost 0.981 ≫ RandomForest 0.91~0.93.
   RandomForest 는 전체 데이터(240만)에서야 부스팅을 따라잡는다(v3 의 0.949 가 그 예).
4. **gender_split 은 작지만 일관된 이득.** 200k 기준 모든 모델에서 분리가 통합보다 약간 높았다
   (RF +0.024, LightGBM +0.003). **최고 = LightGBM gender_split, Test Macro-F1 ≈ 0.982.**
5. **약점은 `수`(수소)와 일부 저등급.** `수` 는 표본 0.6%(test 193마리)로 macro-F1 0.64 에 그친다(통합 fallback).
   등급 중에선 `3A`(0.93)·`2A`(0.97)가 상대적으로 낮다. 개선하려면 이들 표본 보강/가중이 필요.

### 전체 데이터로 재현
맨 위 `cfg` 에서 `use_sample=False` 로 바꾸고 전체 실행 → `model_v4_outputs/` 에 지표·예측·혼동행렬이 저장된다.
(RandomForest 는 전체에서 수십 분 소요. 시간이 없으면 LightGBM 만 돌리는 것을 권장.)
